In [7]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
from src.models import RandomForest, LogisticRegression, KNN, DecisionTree, Metrics, Adam
import numpy as np
import math
import time

In [ ]:
# 1. Load dữ liệu
raw_data = np.genfromtxt('../data/processed/processed_data.csv', delimiter=',', dtype=str, skip_header=1)
feature_names = np.genfromtxt('../data/processed/processed_data.csv', delimiter=',', dtype=str, max_rows=1)

data_numeric = raw_data.astype(float)


# Xáo trộn dữ liệu (Shuffle)
np.random.seed(42) 
np.random.shuffle(data_numeric)

# Tách X và y
X = data_numeric[:, :-1]
y = data_numeric[:, -1]


In [4]:
def manual_train_test_split(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    n_samples = X.shape[0]
    n_test = int(n_samples * test_size)
    
    # Tạo danh sách index ngẫu nhiên
    indices = np.random.permutation(n_samples)
    
    # Cắt index thành 2 phần
    test_indices = indices[:n_test]
    train_indices = indices[n_test:]
    
    # Indexing mảng để lấy dữ liệu
    X_train, X_test = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]
    
    return X_train, X_test, y_train, y_test

# Áp dụng
X_train, X_test, y_train, y_test = manual_train_test_split(X, y, test_size=0.2)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 15327, Test size: 3831


In [8]:
print("\nTest các thuật toán")

# Định nghĩa danh sách thí sinh
models = {
    "Logistic Regression": LogisticRegression(n_iters=3000, optimizer=Adam(lr=0.01)), 
    "K-Nearest Neighbors": KNN(k=5), 
    "Decision Tree      ": DecisionTree(max_depth=10, min_samples_split=10),
    "Random Forest      ": RandomForest(n_trees=5, max_depth=8) 
}

best_score = 0
best_model_name = ""
best_model_instance = None #

print(f"{'Tên Model':<25} | {'Accuracy':<10} | {'Thời gian (s)':<10}")
print("-" * 55)

for name, model in models.items():
    start_time = time.time()
    
    # Huấn luyện
    model.fit(X_train, y_train)
    
    # Đánh giá sơ bộ bằng Accuracy
    acc = model.score(X_test, y_test)
    
    end_time = time.time()
    duration = end_time - start_time
    
    print(f"{name:<25} | {acc:.4f}     | {duration:.2f}")
    
    # So sánh để tìm model tốt nhất
    if acc > best_score:
        best_score = acc
        best_model_name = name
        best_model_instance = model

print("-" * 55)
print(f"Model tốt nhất {best_model_name} (Acc: {best_score:.4f})")


Test các thuật toán
Tên Model                 | Accuracy   | Thời gian (s)
-------------------------------------------------------
Logistic Regression       | 0.7726     | 4.53
K-Nearest Neighbors       | 0.7674     | 1.28
Decision Tree             | 0.7708     | 2.32
Random Forest             | 0.7732     | 0.78
-------------------------------------------------------
Model tốt nhất Random Forest       (Acc: 0.7732)


In [9]:


print(f"\n{'='*40}")
print(f"BÁO CÁO CHI TIẾT CHO: {best_model_name}")
print(f"{'='*40}")

# 1. Dự đoán lại trên tập Test bằng model tốt nhất
y_pred_final = best_model_instance.predict(X_test)

# 2. Tính toán các chỉ số 
accuracy = Metrics.accuracy(y_test, y_pred_final)
precision = Metrics.precision(y_test, y_pred_final)
recall = Metrics.recall(y_test, y_pred_final)
f1 = Metrics.f1_score(y_test, y_pred_final)

# 3. Lấy Confusion Matrix
tp, tn, fp, fn = Metrics.confusion_matrix(y_test, y_pred_final)

# 4. In kết quả
print(f"\n CHỈ SỐ TỔNG HỢP:")
print(f"   - Accuracy  (Độ chính xác): {accuracy:.4f}")
print(f"   - Precision (Độ chuẩn xác): {precision:.4f}")
print(f"   - Recall    (Độ nhạy)     : {recall:.4f}")
print(f"   - F1 Score  (Điểm cân bằng): {f1:.4f}")

print(f"\n MA TRẬN NHẦM LẪN (CONFUSION MATRIX):")
print(f"{'':>20} {'Dự đoán: 0 (Ở lại)':<20} {'Dự đoán: 1 (Rời đi)':<20}")
print(f"{'Thực tế: 0':>20} {tn:<20} {fp:<20}")
print(f"{'Thực tế: 1':>20} {fn:<20} {tp:<20}")

# Giải thích nhanh ý nghĩa con số
print("-" * 60)
print(f" Giải thích:")
print(f"- Có {tp} trường hợp dự đoán ĐÚNG là nhân viên sẽ nghỉ việc.")
print(f"- Có {tn} trường hợp dự đoán ĐÚNG là nhân viên sẽ ở lại.")
print(f"- Có {fp} trường hợp báo động GIẢ (dự đoán nghỉ nhưng thực tế ở lại).")
print(f"- Có {fn} trường hợp bỏ SÓT (dự đoán ở lại nhưng thực tế nghỉ).")


BÁO CÁO CHI TIẾT CHO: Random Forest      

 CHỈ SỐ TỔNG HỢP:
   - Accuracy  (Độ chính xác): 0.7732
   - Precision (Độ chuẩn xác): 0.6000
   - Recall    (Độ nhạy)     : 0.3587
   - F1 Score  (Điểm cân bằng): 0.4490

 MA TRẬN NHẦM LẪN (CONFUSION MATRIX):
                     Dự đoán: 0 (Ở lại)   Dự đoán: 1 (Rời đi) 
          Thực tế: 0 2608                 236                 
          Thực tế: 1 633                  354                 
------------------------------------------------------------
 Giải thích:
- Có 354 trường hợp dự đoán ĐÚNG là nhân viên sẽ nghỉ việc.
- Có 2608 trường hợp dự đoán ĐÚNG là nhân viên sẽ ở lại.
- Có 236 trường hợp báo động GIẢ (dự đoán nghỉ nhưng thực tế ở lại).
- Có 633 trường hợp bỏ SÓT (dự đoán ở lại nhưng thực tế nghỉ).
